<a href="https://colab.research.google.com/github/durgesh-js/RAGX/blob/main/RAGX.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAGX — Retrieval-Augmented Intelligence Engine

RAGX is a Retrieval-Augmented Generation (RAG) system designed to answer questions by retrieving relevant information from Wikipedia and synthesizing grounded responses using the Gemini language model. It demonstrates a practical application of combining information retrieval with large language models to produce accurate and well-sourced answers.

## 1. Project Overview

Retrieval-Augmented Generation (RAG) enhances the capabilities of large language models (LLMs) by providing them with external, up-to-date knowledge during the generation process. This mitigates issues like factual inaccuracies and outdated information often found in LLMs trained on static datasets.

RAGX implements a RAG pipeline where a user's question first triggers a search for relevant articles on Wikipedia. These articles serve as the knowledge base from which a generative AI model, in this case, Google's Gemini, formulates an answer grounded in the retrieved content. This approach ensures that responses are factual, current, and directly supported by verifiable sources, addressing the LLM's inherent limitations in accessing real-time or domain-specific information.

## 2. System Architecture

The RAGX system coordinates information retrieval and generative AI through a sequential flow:

User Question $\rightarrow$ Wikipedia Search $\rightarrow$ Article Retrieval $\rightarrow$ Text Cleaning & Chunking $\rightarrow$ Sentence Transformer Embeddings $\rightarrow$ FAISS Indexing $\rightarrow$ Semantic Retrieval $\rightarrow$ Context Assembly $\rightarrow$ Gemini Generation $\rightarrow$ Grounded Answer + Source Citations

## 3. Environment Setup

This section handles the installation of necessary libraries and the import of modules required for the RAGX system.

In [1]:
!pip -q install google-generativeai sentence-transformers faiss-cpu requests beautifulsoup4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 65.9 MB/s eta 0:00:00


In [2]:
import requests
import numpy as np
import faiss
import re

from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from google import genai

In [3]:
from google import genai
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.6-flash"

print("Gemini client ready!")

Gemini client ready!


In [4]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully!


## 4. Wikipedia Knowledge Retrieval

These functions search Wikipedia for pages matching a query and retrieve their full text content to build the external knowledge source.

In [5]:
import time
import requests

def search_wikipedia(query, limit=5):
    url = "https://en.wikipedia.org/w/rest.php/v1/search/page"

    params = {
        "q": query,
        "limit": limit
    }

    headers = {
        "User-Agent": "RAGX/1.0 (educational project)"
    }

    response = requests.get(
        url,
        params=params,
        headers=headers,
        timeout=10
    )

    response.raise_for_status()

    data = response.json()

    pages = []

    for page in data.get("pages", []):
        pages.append({
            "title": page.get("title", ""),
            "description": page.get("description", ""),
            "excerpt": page.get("excerpt", ""),
            "key": page.get("key", "")
        })

    return pages

In [6]:
def simplify_query(question):
    query = question.lower().strip()

    phrases = [
        "how does ",
        "how do ",
        "how ",
        "what is ",
        "what are ",
        "explain ",
        "describe ",
        "tell me about "
    ]

    for phrase in phrases:
        if query.startswith(phrase):
            query = query[len(phrase):]
            break

    query = query.replace(" works", "")
    query = query.strip()

    return query

In [7]:
import time
import requests

def get_wikipedia_page(title, retries=3):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": 1,
        "exsectionformat": "plain",
        "titles": title,
        "format": "json",
        "redirects": 1
    }

    headers = {
        "User-Agent": "WikiMind-RAG/1.0 (educational project)"
    }

    for attempt in range(retries):

        response = requests.get(
            url,
            params=params,
            headers=headers,
            timeout=15
        )

        if response.status_code == 429:
            wait_time = 5 * (attempt + 1)
            print(f"Wikipedia rate limit reached. Waiting {wait_time}s...")
            time.sleep(wait_time)
            continue

        response.raise_for_status()

        data = response.json()
        pages = data["query"]["pages"]
        page = next(iter(pages.values()))

        return {
            "title": page.get("title", title),
            "text": page.get("extract", ""),
            "url": "https://en.wikipedia.org/wiki/" + title.replace(" ", "_")
        }

    return {
        "title": title,
        "text": "",
        "url": "https://en.wikipedia.org/wiki/" + title.replace(" ", "_")
    }

In [8]:
def retrieve_wikipedia_pages(query, num_pages=5):

    search_results = search_wikipedia(
        query,
        limit=num_pages
    )

    documents = []

    for result in search_results:

        try:
            page = get_wikipedia_page(result["title"])

            if len(page["text"]) > 100:
                documents.append(page)

            time.sleep(1)

        except Exception as e:
            print("Error retrieving:", result["title"])
            print(e)

    return documents

## 5. Text Cleaning & Chunking

Long Wikipedia pages are cleaned and split into smaller, overlapping chunks so that specific, relevant passages can be matched and retrieved.

In [9]:
import re

def clean_text(text):

    text = re.sub(r"\s+", " ", text)

    return text.strip()


def chunk_text(text, chunk_size=500, overlap=100):

    text = clean_text(text)

    chunks = []

    start = 0

    while start < len(text):

        end = start + chunk_size

        chunk = text[start:end]

        if len(chunk) > 100:
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [10]:
def create_chunks(documents):

    all_chunks = []

    for doc in documents:

        chunks = chunk_text(doc["text"])

        for i, chunk in enumerate(chunks):

            all_chunks.append({
                "text": chunk,
                "title": doc["title"],
                "url": doc["url"],
                "chunk_id": i
            })

    return all_chunks

## 6. Semantic Embeddings

This function converts each text chunk into a high-dimensional vector representation capturing its semantic meaning.

In [11]:
def create_embeddings(chunks):

    texts = [chunk["text"] for chunk in chunks]

    embeddings = embedding_model.encode(
        texts,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return embeddings

## 7. FAISS Semantic Retrieval

These functions index the chunk embeddings and retrieve the top matching passages for a given question using vector similarity.

In [12]:
import faiss

def create_faiss_index(embeddings):

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(
        embeddings.astype("float32")
    )

    return index

In [13]:
def retrieve_relevant_chunks(
    question,
    chunks,
    index,
    top_k=8,
    min_score=0.35
):

    question_embedding = embedding_model.encode(
        [question],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        question_embedding.astype("float32"),
        top_k
    )

    results = []

    for score, idx in zip(scores[0], indices[0]):

        if idx != -1 and score >= min_score:

            result = chunks[idx].copy()
            result["score"] = float(score)

            results.append(result)

    return results

## 8. Grounded Answer Generation

These functions format the retrieved passages into a structured prompt and instruct Gemini to generate an answer based only on this provided context.

In [14]:
def build_prompt(question, retrieved_chunks):

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks):

        context_parts.append(
            f"""
SOURCE {i + 1}
Title: {chunk['title']}
URL: {chunk['url']}

Content:
{chunk['text']}
"""
        )

    context = "\n".join(context_parts)

    prompt = f"""
You are RAGX, a retrieval-grounded AI assistant.

Answer the user's question using only the retrieved Wikipedia context.

Rules:
1. Use the retrieved context as the primary source.
2. Do not invent unsupported facts.
3. Give a concise and clear answer.
4. If the context does not contain enough information, say so.
5. Do not mention information that is not supported by the context.

Retrieved Context:
{context}

User Question:
{question}

Answer:
"""

    return prompt

In [15]:
def generate_answer(question, retrieved_chunks):

    prompt = build_prompt(
        question,
        retrieved_chunks
    )

    response = client.interactions.create(
        model=MODEL_NAME,
        input=prompt
    )

    return response.output_text

In [16]:
def display_sources(retrieved_chunks):

    print("\n" + "=" * 70)
    print("SOURCES")
    print("=" * 70)

    seen = set()

    for chunk in retrieved_chunks:

        if chunk["title"] not in seen:

            print(f"• {chunk['title']}")
            print(f"  {chunk['url']}")
            print()

            seen.add(chunk["title"])

## 9. End-to-End RAG Pipeline

This main function integrates the retrieval, indexing, search, and generation steps into a single executable pipeline.

In [17]:
def wiki_rag(question):

    print("🔎 Searching Wikipedia...")

    documents = retrieve_wikipedia_pages(
        question,
        num_pages=5
    )

    if not documents:
        return "No relevant Wikipedia pages were found."

    print(f"📚 Retrieved {len(documents)} Wikipedia pages")

    chunks = create_chunks(documents)

    print(f"✂️ Created {len(chunks)} chunks")

    embeddings = create_embeddings(chunks)

    index = create_faiss_index(embeddings)

    retrieved = retrieve_relevant_chunks(
        question,
        chunks,
        index,
        top_k=5
    )

    print("🧠 Retrieved top 5 relevant chunks")

    answer = generate_answer(
        question,
        retrieved
    )

    print("\n" + "=" * 70)
    print("WIKIMIND ANSWER")
    print("=" * 70)

    print(answer)

    display_sources(retrieved)

    return answer

## 10. Retrieval Inspection

This utility displays the retrieved text chunks and their similarity scores, making the intermediate retrieval step transparent and verifiable.

In [18]:
def show_retrieved_chunks(retrieved):

    print("\n" + "=" * 70)
    print("RETRIEVED KNOWLEDGE")
    print("=" * 70)

    for i, chunk in enumerate(retrieved):

        print(f"\n[{i + 1}] {chunk['title']}")
        print("Similarity:", round(chunk["score"], 4))
        print("Chunk ID:", chunk["chunk_id"])
        print("Text:", chunk["text"][:300])

In [19]:
# Example of retrieval for inspection
question_for_inspection = "What is artificial intelligence?"

# Run a partial pipeline to get retrieved chunks
documents_inspect = retrieve_wikipedia_pages(question_for_inspection, num_pages=3)
chunks_inspect = create_chunks(documents_inspect)
embeddings_inspect = create_embeddings(chunks_inspect)
index_inspect = create_faiss_index(embeddings_inspect)
retrieved_inspect = retrieve_relevant_chunks(question_for_inspection, chunks_inspect, index_inspect, top_k=5)

show_retrieved_chunks(retrieved_inspect)


RETRIEVED KNOWLEDGE

[1] Artificial intelligence
Similarity: 0.8554
Chunk ID: 0
Text: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that devel

[2] Artificial intelligence
Similarity: 0.6585
Chunk ID: 191
Text: of its history. The unprecedented success of statistical machine learning in the 2010s eclipsed all other approaches (so much so that some sources, especially in the business world, use the term "artificial intelligence" to mean "machine learning with neural networks"). This approach is mostly sub-s

[3] Artificial general intelligence
Similarity: 0.6555
Chunk ID: 8
Text:  "We cannot yet characterize in general what kinds of computational procedures we want to call intelligent." Intelligence traits Researchers generally hold that a system

## 11. Evaluation

This run demonstrates the complete pipeline output, answering a query and showing the grounded result alongside its verified Wikipedia source URLs.

## 12. Interactive RAGX Demonstration

This interactive console lets you input custom questions to run the full RAGX pipeline and view the generated answers with source citations.

In [20]:
while True:

    question = input(
        "\n╭──────────────────────────────────────────────╮\n"
        "│              ⚡ RAGX QUERY CONSOLE           │\n"
        "╰──────────────────────────────────────────────╯\n"
        "  Enter your question (or type 'exit'): "
    )

    if question.strip().lower() == "exit":
        print("\n⚡ RAGX session ended.")
        break

    print("\n" + "─" * 65)
    print("🔎 RETRIEVING KNOWLEDGE FROM WIKIPEDIA...")
    print("─" * 65)

    search_query = simplify_query(question)

    documents = retrieve_wikipedia_pages(
        search_query,
        num_pages=5
    )

    if not documents:
        print("⚠️ No relevant Wikipedia pages were found.")
        continue

    print(f"✓ {len(documents)} Wikipedia sources retrieved")

    chunks = create_chunks(documents)
    print(f"✓ {len(chunks)} knowledge segments created")

    embeddings = create_embeddings(chunks)
    index = create_faiss_index(embeddings)

    retrieved = retrieve_relevant_chunks(
        question,
        chunks,
        index,
        top_k=8,
        min_score=0.35
    )

    if not retrieved:
        print("⚠️ No sufficiently relevant information was found.")
        continue

    print("✓ Semantic retrieval completed")

    answer = generate_answer(
        question,
        retrieved
    )

    print("\n" + "═" * 65)
    print("⚡ RAGX | INTELLIGENCE ENGINE")
    print("═" * 65)

    print("\n" + answer)

    print("\n" + "─" * 65)
    print("📚 KNOWLEDGE SOURCES")
    print("─" * 65)

    seen = set()

    for i, chunk in enumerate(retrieved, 1):

        if chunk["title"] not in seen:
            print(f"\n[{i}] {chunk['title']}")
            print(f"    🔗 {chunk['url']}")
            seen.add(chunk["title"])

    print("\n" + "═" * 65)


╭──────────────────────────────────────────────╮
│              ⚡ RAGX QUERY CONSOLE           │
╰──────────────────────────────────────────────╯
  Enter your question (or type 'exit'): How does quantum computing work?

─────────────────────────────────────────────────────────────────
🔎 RETRIEVING KNOWLEDGE FROM WIKIPEDIA...
─────────────────────────────────────────────────────────────────
✓ 5 Wikipedia sources retrieved
✓ 314 knowledge segments created
✓ Semantic retrieval completed

═════════════════════════════════════════════════════════════════
⚡ RAGX | INTELLIGENCE ENGINE
═════════════════════════════════════════════════════════════════

Based on the provided context, quantum computing works through the following core principles and paradigms:

* **Basic Unit (Qubits):** Instead of classical binary bits, quantum computing uses **qubits** (quantum bits). Unlike a classical bit, a qubit can exist in a linear combination of states known as a **quantum superposition**.
* **Quantum P